# Update a CATIA parameter

Push a new parameter value into a CATIA model on the Istari Digital Platform with **`@istari:update_parameters`**.

You will:

1. Connect to the Istari Digital Platform
2. Set the CATIA **resource ID**, parameter path, and target value
3. Submit `@istari:update_parameters` and wait for completion

Extracted from [CAD parameter validation](../validation/validation.ipynb) (§7). For 3DEXPERIENCE connect + extract, see [Connect resources — CATIA 3DEXPERIENCE](../resources/connect-resources-catia-3dx.ipynb).

### Prerequisites

- **`istari_labs_helpers`** kernel and [`samples/.env`](../.env) — see [Chaining jobs](../chaining_jobs.ipynb).
- A CATIA **Model** resource already registered on the platform (upload or connected pointer).
- Access to **`@istari:update_parameters`** with tool **`dassault_catia_v5`** on a Windows agent.

### Running order

Run top to bottom. Edit **Prep** before submitting the job.

> **Pause in the web app:** after §3, open **Jobs** to watch `@istari:update_parameters` run on the agent.

## 1 · Connect

In [ ]:
from pathlib import Path

from istari_labs_helpers import IstariPlatform, JobDefinition

NOTEBOOK_DIR = Path.cwd()
platform = IstariPlatform.from_env(str(NOTEBOOK_DIR.parent / ".env"))

current_user = platform.client.get_current_user()
print(platform)
print(f"User ID: {current_user.id}")

## 2 · Prep

Paste the CATIA model **resource ID** from the Istari Digital web app (**Files** / Models). Set the parameter path and value to update.

In [ ]:
RESOURCE_ID = input("CATIA resource ID: ").strip()
if not RESOURCE_ID:
    raise ValueError("RESOURCE_ID is required — paste the model UUID from the web app.")

# Path: CATEGORY\PARAMETER_NAME (must match the model). Value: number + unit, no space.
PARAMETER_PATH = r"SA ISTARI_ONE\WING.2\WING_LENGTH"
# PARAMETER_PATH = r"WING\WING_LENGTH"
PARAMETER_VALUE = "2300mm"

TOOL_NAME = "dassault_catia_v5"
TOOL_VERSION = "6R2023"
OPERATING_SYSTEM = "Windows 10"

print(f"Resource ID: {RESOURCE_ID}")
print(f"Parameter:   {PARAMETER_PATH} → {PARAMETER_VALUE}")

## 3 · Submit `@istari:update_parameters`

Resolve the resource, submit the update job, and poll until it completes.

In [ ]:
catia_model = platform.get_model(RESOURCE_ID)

update_def = JobDefinition(
    function="@istari:update_parameters",
    tool_name=TOOL_NAME,
    tool_version=TOOL_VERSION,
    operating_system=OPERATING_SYSTEM,
    # Nested dict under "parameters" — agent wraps values; keep PARAMETER_VALUE as "1750mm".
    parameters={
        "parameters": {
            PARAMETER_PATH: PARAMETER_VALUE,
        },
    },
)

update_job = catia_model.submit_job(update_def)
print(f"Submitted update job {update_job.id}; polling...")

update_job.wait(
    timeout=900,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"{PARAMETER_PATH} set to {PARAMETER_VALUE} — job {update_job.id} completed")

## Learn more

- [CAD parameter validation](../validation/validation.ipynb) — compare Cameo requirements to CATIA parameters, then optionally update
- [Connect resources — CATIA 3DEXPERIENCE](../resources/connect-resources-catia-3dx.ipynb) — register a connected CATIA pointer and extract
- [Basic CATIA `.CATPart` extraction](../../integrations/basic_catia_catpart_extraction.ipynb) — upload and extract a `.CATPart`